# Multi-Agent Research Lab — Demo Notebook

**Student**: Ho Dac Toan — 2A202600057 | **Lab**: 20

Notebook này demo luồng chạy của hệ thống multi-agent research. Production code nằm trong `src/` — notebook chỉ gọi vào đó.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

# Activate tracing FIRST — must happen before any workflow import
import multi_agent_research_lab.observability.tracing as tracing

from multi_agent_research_lab.core.config import get_settings
settings = get_settings()

print(f"Model          : {settings.openai_model}")
print(f"Max iterations : {settings.max_iterations}")
print(f"LangSmith      : {'active' if tracing.langsmith_active else 'inactive'}")
print(f"Langfuse       : {'active' if tracing.langfuse_active else 'inactive'}")

## 1. Baseline — Single LLM Call

In [ ]:
from time import perf_counter

from multi_agent_research_lab.core.schemas import ResearchQuery
from multi_agent_research_lab.core.state import ResearchState
from multi_agent_research_lab.services.llm_client import LLMClient

QUERY = "Research GraphRAG state-of-the-art and write a 500-word summary"

llm = LLMClient()
t0 = perf_counter()
resp = llm.complete(
    system_prompt="You are a research assistant. Answer as comprehensively as possible.",
    user_prompt=QUERY,
)
baseline_latency = perf_counter() - t0

print(f"Latency : {baseline_latency:.2f}s")
print(f"Tokens  : {resp.input_tokens} in / {resp.output_tokens} out")
print(f"Cost    : ${resp.cost_usd:.6f}")
print()
print(resp.content[:600], "...")

## 2. Multi-Agent Pipeline

In [ ]:
from multi_agent_research_lab.graph.workflow import MultiAgentWorkflow

state = ResearchState(request=ResearchQuery(query=QUERY))
workflow = MultiAgentWorkflow()

t0 = perf_counter()
result = workflow.run(state)
multi_latency = perf_counter() - t0

print(f"Latency        : {multi_latency:.2f}s")
print(f"Agents invoked : {[r.agent.value for r in result.agent_results]}")
print(f"Route history  : {result.route_history}")
print(f"Sources found  : {len(result.sources)}")
print(f"Total cost     : ${sum(r.metadata.get('cost_usd', 0) for r in result.agent_results):.6f}")

In [ ]:
# Show final answer
print(result.final_answer)

## 3. Agent-by-Agent Trace

In [ ]:
import textwrap

for r in result.agent_results:
    cost = r.metadata.get('cost_usd', 0)
    tokens_in = r.metadata.get('input_tokens', '-')
    tokens_out = r.metadata.get('output_tokens', '-')
    verdict = r.metadata.get('verdict', '')
    print(f"[{r.agent.value:10s}] tokens={tokens_in}/{tokens_out}  cost=${cost:.6f}  {verdict}")
    snippet = textwrap.shorten(r.content, width=120, placeholder=" ...")
    print(f"  {snippet}")
    print()

## 4. Sources Retrieved

In [ ]:
for i, src in enumerate(result.sources, 1):
    print(f"[{i}] {src.title}")
    print(f"    {src.url}")
    print(f"    Score: {src.score:.3f}")
    print()

## 5. Baseline vs Multi-Agent — Comparison

In [ ]:
import re

def quality_score(text: str, sources: int) -> float:
    words = len(text.split())
    citations = len(re.findall(r'\[\d+\]', text))
    return round(min(words / 500 * 5, 5.0) + min(citations * 0.5, 5.0), 2)

baseline_quality = quality_score(resp.content, 0)
multi_quality    = quality_score(result.final_answer or "", len(result.sources))
multi_cost       = sum(r.metadata.get('cost_usd', 0) for r in result.agent_results)

print(f"{'Metric':<22} {'Baseline':>12} {'Multi-Agent':>14}")
print("-" * 50)
print(f"{'Latency (s)':<22} {baseline_latency:>12.2f} {multi_latency:>14.2f}")
print(f"{'Cost (USD)':<22} {resp.cost_usd:>12.6f} {multi_cost:>14.6f}")
print(f"{'Quality (0-10)':<22} {baseline_quality:>12} {multi_quality:>14}")
print(f"{'Sources':<22} {'0':>12} {len(result.sources):>14}")
print(f"{'Citations':<22} {'0':>12} {len(re.findall(chr(91)+r'\d+'+chr(93), result.final_answer or '')):>14}")

## 6. Quality Score Breakdown

In [ ]:
def quality_breakdown(label: str, text: str) -> None:
    words = len(text.split())
    citations = len(re.findall(r'\[\d+\]', text))
    length_pts  = min(words / 500 * 5, 5.0)
    cite_pts    = min(citations * 0.5, 5.0)
    total       = length_pts + cite_pts
    print(f"{label}")
    print(f"  Words       : {words:>5}  -> {length_pts:.1f}/5.0 pts")
    print(f"  Citations   : {citations:>5}  -> {cite_pts:.1f}/5.0 pts")
    print(f"  Total score : {total:.1f}/10.0")
    print()

quality_breakdown("Baseline", resp.content)
quality_breakdown("Multi-Agent", result.final_answer or "")

## 7. Flush Traces

In [ ]:
tracing.flush_langfuse()
print("Traces flushed.")